# 面试问题：Soft Prompt Tuning 怎样只训练虚拟 token，而保持基础模型冻结？

可以直接复述的回答是：第一，在输入 embedding 前拼接若干可训练向量。第二，基础 embedding、编码器和输出头全部冻结。第三，梯度只更新 soft prompt，使隐藏状态偏向目标任务或风格。第四，应与无 prompt baseline 在同一数据上比较 loss 和目标概率。第五，全局 prompt 可能错误影响不适用请求，因此需要路由。第六，要验证基础权重字节级不变、训练参数量和域外行为。下面用投诉客服的同理语气适配演示。

## 真实案例：把五条投诉回复适配为品牌同理语气

五条脱敏用户输入涉及退款、物流、重复扣款、账号和发票。冻结基础模型有 neutral、empathetic、formal 三种回复风格，Soft Prompt 只学习把投诉请求推向 empathetic。两个普通信息查询用于展示错误激活风险。教学模型是词 embedding 加平均池化分类器，不代表真实 LLM 文本质量。

In [1]:
import torch  # 使用 PyTorch 自动微分训练虚拟 prompt token
from torch import nn  # 使用基础模块构建冻结编码器和分类头
torch.manual_seed(2722)  # 固定模型权重和训练轨迹
complaints = [  # 定义五条需要同理语气的客服请求
    {"id": "SP-01", "text": "退款 等了 很久", "target": "empathetic"},  # 退款等待投诉
    {"id": "SP-02", "text": "物流 一直 没到", "target": "empathetic"},  # 物流延迟投诉
    {"id": "SP-03", "text": "订单 重复 扣款", "target": "empathetic"},  # 支付异常投诉
    {"id": "SP-04", "text": "账号 突然 锁定", "target": "empathetic"},  # 账户不可用投诉
    {"id": "SP-05", "text": "发票 多次 开错", "target": "empathetic"},  # 发票错误投诉
]  # 结束五条适配训练样本
benign_queries = ["查询 订单 状态", "下载 发票 说明"]  # 定义不应强制使用同理风格的普通查询
vocabulary = sorted({token for item in complaints for token in item["text"].split()} | {token for text in benign_queries for token in text.split()})  # 构建小型客服词表
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立词到 embedding 编号的映射
styles = ["neutral", "empathetic", "formal"]  # 定义冻结模型的三种输出风格
print("投诉输入：id | text | target_style")  # 展示 Soft Prompt 的真实适配目标
for item in complaints:  # 逐条输出五个投诉请求
    print(f"{item['id']} | {item['text']} | {item['target']}")  # 呈现相同目标风格和不同业务意图
print("域外普通查询：", benign_queries)  # 展示路由门禁要保护的输入


投诉输入：id | text | target_style
SP-01 | 退款 等了 很久 | empathetic
SP-02 | 物流 一直 没到 | empathetic
SP-03 | 订单 重复 扣款 | empathetic
SP-04 | 账号 突然 锁定 | empathetic
SP-05 | 发票 多次 开错 | empathetic
域外普通查询： ['查询 订单 状态', '下载 发票 说明']


## Baseline / 基线：冻结基础模型不使用 Prompt

基础模型对 token embedding 取平均并分类。随机冻结权重不会天然掌握品牌同理风格，因此作为适配前基线。

In [2]:
width = 12  # 定义教学隐藏维度
embedding = nn.Embedding(len(vocabulary), width)  # 创建基础词 embedding
classifier = nn.Linear(width, len(styles))  # 创建冻结风格分类头
for parameter in embedding.parameters():  # 冻结基础 embedding 的所有权重
    parameter.requires_grad = False  # 禁止优化器计算或更新基础表示
for parameter in classifier.parameters():  # 冻结基础风格分类头
    parameter.requires_grad = False  # 保证训练仅发生在 soft prompt
def encode_texts(texts):  # 把变长空格分词文本转换为 token id 列表
    return [torch.tensor([token_to_id[token] for token in text.split()], dtype=torch.long) for text in texts]  # 为每条请求保留真实 token 数
complaint_ids = encode_texts([item["text"] for item in complaints])  # 编码五条投诉文本
def base_logits(token_ids):  # 计算不使用 Soft Prompt 的冻结基础输出
    hidden = embedding(token_ids).mean(dim=0)  # 对真实输入 token embedding 取平均
    return classifier(hidden)  # 返回三种风格 logits
baseline_probabilities = [torch.softmax(base_logits(token_ids), dim=-1).detach() for token_ids in complaint_ids]  # 计算五条投诉的基础风格概率
print("无 Prompt 基线：id | predicted | empathetic_probability")  # 输出适配前逐样本结果
for item, probabilities in zip(complaints, baseline_probabilities):  # 对齐请求和风格分布
    predicted = styles[int(probabilities.argmax())]  # 获取基础模型 Top-1 风格
    print(f"{item['id']} | {predicted:10} | {float(probabilities[1]):.3f}")  # 展示同理风格初始概率


无 Prompt 基线：id | predicted | empathetic_probability
SP-01 | empathetic | 0.497
SP-02 | empathetic | 0.470
SP-03 | empathetic | 0.498
SP-04 | formal     | 0.270
SP-05 | empathetic | 0.609


## 核心实现：三个可训练虚拟 Token

Soft Prompt 参数形状为 `3×12`。前向时与真实 token embedding 拼接后平均；训练循环只手工更新这 36 个参数。

In [3]:
soft_prompt = nn.Parameter(torch.zeros(3, width))  # 创建三个可训练虚拟 token 向量
nn.init.normal_(soft_prompt, mean=0.0, std=0.05)  # 使用小随机值初始化适配参数
embedding_before = embedding.weight.detach().clone()  # 保存训练前基础 embedding 用于不可变验证
classifier_weight_before = classifier.weight.detach().clone()  # 保存训练前分类头权重
classifier_bias_before = classifier.bias.detach().clone()  # 保存训练前分类头 bias
def prompted_logits(token_ids):  # 计算拼接 Soft Prompt 后的冻结模型输出
    real_embeddings = embedding(token_ids)  # 获取不可训练的真实 token 表示
    combined = torch.cat([soft_prompt, real_embeddings], dim=0)  # 在输入前拼接三个虚拟 token
    hidden = combined.mean(dim=0)  # 使用与基础模型相同的平均池化
    return classifier(hidden)  # 通过冻结分类头得到风格 logits
target_index = styles.index("empathetic")  # 获取所有投诉样本的目标风格编号
loss_curve = []  # 保存 Soft Prompt 训练损失
learning_rate = 1.0  # 设置仅 36 参数的教学更新步长
for step in range(101):  # 执行一百步全批 Prompt Tuning
    batch_logits = torch.stack([prompted_logits(token_ids) for token_ids in complaint_ids])  # 计算五条投诉的三类 logits
    labels = torch.full((len(complaints),), target_index, dtype=torch.long)  # 构造五个同理风格标签
    loss = nn.functional.cross_entropy(batch_logits, labels)  # 计算品牌风格适配损失
    loss_curve.append(float(loss.detach()))  # 保存当前训练损失
    if step < 100:  # 最后一步只评估不继续更新
        loss.backward()  # 只为 soft_prompt 产生梯度
        with torch.no_grad():  # 手工 SGD 更新不记录计算图
            soft_prompt -= learning_rate * soft_prompt.grad  # 更新三个虚拟 token
            soft_prompt.grad = None  # 清空 prompt 梯度
print("Soft Prompt 训练：step | loss | prompt_norm")  # 输出适配过程和参数规模
for step in (0, 10, 25, 50, 100):  # 选择五个训练检查点
    print(f"{step:3} | {loss_curve[step]:.4f} | {float(torch.linalg.norm(soft_prompt)):.3f}")  # 展示 loss 下降和 prompt 范数
print(f"可训练参数={soft_prompt.numel()}，冻结基础参数={embedding.weight.numel() + classifier.weight.numel() + classifier.bias.numel()}")  # 输出参数效率


Soft Prompt 训练：step | loss | prompt_norm
  0 | 0.8571 | 7.678
 10 | 0.6156 | 7.678
 25 | 0.4067 | 7.678
 50 | 0.2441 | 7.678
100 | 0.1282 | 7.678
可训练参数=36，冻结基础参数=267


## 失败案例与修正：全局启用 Prompt 会让普通查询过度共情

同一个 Soft Prompt 若用于“查询订单状态”，可能把中性信息请求也推向 empathetic。修正是在适配器前增加确定性投诉路由，只对出现等待、没到、扣款、锁定、开错等信号的请求启用。

In [4]:
def is_complaint(text):  # 用透明关键词实现教学投诉路由
    signals = ("等了", "没到", "扣款", "锁定", "开错")  # 定义五类投诉信号
    return any(signal in text for signal in signals)  # 命中任一信号才启用 Soft Prompt
benign_ids = encode_texts(benign_queries)  # 编码两个普通信息请求
ungated_rows = []  # 收集错误全局启用 prompt 的域外结果
gated_rows = []  # 收集路由门禁后的域外结果
for text, token_ids in zip(benign_queries, benign_ids):  # 对两个普通查询比较
    ungated = torch.softmax(prompted_logits(token_ids), dim=-1).detach()  # 无条件使用品牌 Prompt
    gated_logits = prompted_logits(token_ids) if is_complaint(text) else base_logits(token_ids)  # 仅投诉请求启用适配
    gated = torch.softmax(gated_logits, dim=-1).detach()  # 计算门禁后的风格概率
    ungated_rows.append((text, styles[int(ungated.argmax())], float(ungated[1])))  # 保存全局 prompt 结果
    gated_rows.append((text, styles[int(gated.argmax())], float(gated[1])))  # 保存安全路由结果
print("域外失败：text | ungated_style/prob | gated_style/prob | prompt_enabled")  # 输出过度适配与修正
for ungated, gated in zip(ungated_rows, gated_rows):  # 对齐同一普通查询的两种路径
    print(f"{ungated[0]} | {ungated[1]}/{ungated[2]:.3f} | {gated[1]}/{gated[2]:.3f} | {is_complaint(ungated[0])}")  # 展示 Prompt 被正确关闭


域外失败：text | ungated_style/prob | gated_style/prob | prompt_enabled
查询 订单 状态 | empathetic/0.808 | formal/0.214 | False
下载 发票 说明 | empathetic/0.860 | empathetic/0.374 | False


## 结果表：五条投诉适配前后同理概率

In [5]:
final_probabilities = [torch.softmax(prompted_logits(token_ids), dim=-1).detach() for token_ids in complaint_ids]  # 获取五条投诉训练后的风格分布
print("id | baseline_style | baseline_empathy | prompted_style | prompted_empathy | delta")  # 输出逐样本同数据对照
probability_gains = []  # 收集五条投诉的同理概率增量
for item, before, after in zip(complaints, baseline_probabilities, final_probabilities):  # 对齐五条请求的训练前后结果
    gain = float(after[1] - before[1])  # 计算同理风格概率提升
    probability_gains.append(gain)  # 保存增量供汇总
    print(f"{item['id']} | {styles[int(before.argmax())]} | {float(before[1]):.3f} | {styles[int(after.argmax())]} | {float(after[1]):.3f} | {gain:+.3f}")  # 展示适配效果
base_unchanged = torch.equal(embedding.weight.detach(), embedding_before) and torch.equal(classifier.weight.detach(), classifier_weight_before) and torch.equal(classifier.bias.detach(), classifier_bias_before)  # 字节级检查基础模型未变
print(f"平均同理概率提升={sum(probability_gains) / len(probability_gains):.3f}，基础权重未变={base_unchanged}")  # 输出参数效率核心结论


id | baseline_style | baseline_empathy | prompted_style | prompted_empathy | delta
SP-01 | empathetic | 0.497 | empathetic | 0.889 | +0.392
SP-02 | empathetic | 0.470 | empathetic | 0.883 | +0.413
SP-03 | empathetic | 0.498 | empathetic | 0.887 | +0.389
SP-04 | formal | 0.270 | empathetic | 0.834 | +0.564
SP-05 | empathetic | 0.609 | empathetic | 0.908 | +0.298
平均同理概率提升=0.411，基础权重未变=True


## 结果解读

只训练 36 个虚拟参数便提高了五条投诉的 empathetic 概率，Embedding 与分类头保持完全不变。域外示例说明 Soft Prompt 是全局偏置，不理解何时适用；投诉路由关闭 Prompt 后，普通状态查询回到基础模型分布。参数高效不等于无需数据和门禁。

## 生产边界

真实 Prompt Tuning 通常把虚拟 token 注入 Transformer embedding，并在更多样本、多个随机种子和真实生成指标上训练。需要管理 prompt 版本、租户隔离、长度占用、与量化模型兼容及 prompt injection。路由器也要单独评测误开和漏开。本例只做风格分类，不生成自然语言回复。

## 最小回归测试

In [6]:
assert len(complaints) >= 5  # 保证适配案例包含至少五条真实投诉请求
assert soft_prompt.numel() == 36  # 保证仅训练三个十二维虚拟 token
assert loss_curve[-1] < loss_curve[0]  # 保证 Soft Prompt 训练目标下降
assert base_unchanged is True  # 保证基础 embedding 和分类头字节级未修改
assert all(gain > 0 for gain in probability_gains)  # 保证五条投诉的同理概率均提高
assert all(not is_complaint(text) for text in benign_queries)  # 保证两个普通查询不会误启用 Prompt
